# SOLiD-PINN v0.10.1a — FEM/FVM verification
CPU · Run all · No Drive or token. 90 sequential stages, 117 comparisons. Synthetic matched equations; independent spatial discretization only.


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess,sys,os,tempfile,json,hashlib
REV="c488e2d7d20f57f2de99ca1ecb1f886bcd95680a"
repo=Path(tempfile.mkdtemp(prefix="solid-fem-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"canonical_fv.py":"ef028c3fe1c07dcf2c9071b4bf2192d35fec3dcbb2e4c6adda688bfbb584fe58","foundation_fem.py":"741174d025c5315217f2a6d308ee6ed4b742834bc7f5e3813c5056c0c3db6351","verify_foundation.py":"897f9c50260e25c66c90243979f358df7687fbacf059e29fc2cb35c68bc56385","test_foundation_fem.py":"f6085a3664867096640367852dd789391fa04e44668374f8070d999712bbe89a","test_canonical_fv.py":"75aa8d846dfa11cfa5e1f426d98a3478cefbc3dbe08b94e00b2c6e466eec56d4"}
for n,h in hashes.items(): assert hashlib.sha256((repo/"scripts"/n).read_bytes()).hexdigest()==h
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_canonical_fv.py","scripts/test_foundation_fem.py"],cwd=repo,env=env,check=True)


## Run matched charge / rest / discharge


In [ ]:
subprocess.run([sys.executable,"-u","scripts/verify_foundation.py"],cwd=repo,env=env,check=True)
d=json.loads((repo/"foundation_verification.json").read_text())
labels=("coarse","axial","radial","time","fine")
assert len(d["rows"])==90 and len(d["pairs"])==117
assert {(x["solver"],x["config"],x["rate"],x["stage"]) for x in d["rows"]}=={(s,l,r,k) for s in ("FVM","FEM") for l in labels for r in (20,10,5) for k in ("charge","rest","discharge")}
fine=[x for x in d["pairs"] if x["kind"]=="cross_solver" and x["comparison"]=="fine"]
refinement=[x for x in d["pairs"] if x["kind"]=="refinement" and x["comparison"] in ("radial->time","time->fine")]
assert len(fine)==9 and len(refinement)==36
d["final_refinement_gate"]=all(x["max_voltage_mv"]<=1 and x["capacity_delta_pct"]<=.5 for x in refinement)
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=8)
print({k:v for k,v in d.items() if k.endswith("_gate")})
print("Fine max voltage delta mV:",max(x["max_voltage_mv"] for x in fine))


## Download full hashed JSON


In [ ]:
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
dest=Path("/content")/f"SOLiD-PINN_v0101a_colab_{sha[:20]}.json"
dest.write_bytes(raw)
assert dest.read_bytes()==raw
print(dest,sha)
files.download(str(dest))
